# WP4 — Partie 2 : Entraînement + Inférence

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


## 2. Chargement et normalisation des paires

In [ ]:
data_train = torch.load('wp4_pairs_train.pt')
data_val   = torch.load('wp4_pairs_val.pt')

# Normalisation z_mae sur le train uniquement — z_llava deja L2-normalise
z_mae_mean = data_train['z_mae'].mean(dim=0)
z_mae_std  = data_train['z_mae'].std(dim=0).clamp(min=1e-6)

z_mae_train = (data_train['z_mae'] - z_mae_mean) / z_mae_std
z_mae_val   = (data_val['z_mae']   - z_mae_mean) / z_mae_std

torch.save({'mean': z_mae_mean, 'std': z_mae_std}, 'wp4_norm_stats.pt')

BATCH_SIZE   = 256
train_ds     = TensorDataset(z_mae_train, data_train['z_llava'])
val_ds       = TensorDataset(z_mae_val,   data_val['z_llava'])
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train : {len(train_ds)} paires')
print(f'Val   : {len(val_ds)} paires')
print(f'z_mae  — mean: {z_mae_train.mean():.4f}, std: {z_mae_train.std():.4f}')
print(f'z_llava — norm moy: {data_train["z_llava"].norm(dim=-1).mean():.4f}')


## 3. Architecture MLP

**Loss cosinus** : `z_llava` est L2-normalisé — la loss cosinus mesure uniquement l'angle entre vecteurs, ce qui est exactement ce qu'on optimise dans un espace L2-normalisé.

In [ ]:
class ProjectionMLP(nn.Module):
    """
    MAE z_full (1024) -> espace LLM LLaVA (4096)
    BatchNorm -> Linear(1024->2048) -> GELU -> Dropout -> LayerNorm -> Linear(2048->4096) -> L2 norm
    """
    def __init__(self, in_dim=1024, hidden_dim=2048, out_dim=4096, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm1d(in_dim),
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, out_dim),
        )
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)

model = ProjectionMLP().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'Parametres : {n_params:,}')


## 4. Entraînement

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

N_EPOCHS     = 200
LR           = 1e-3
WEIGHT_DECAY = 1e-3
PATIENCE     = 20

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
history   = {'train_loss': [], 'val_loss': [], 'val_cos': []}
best_cos, best_epoch, patience_counter = 0., 0, 0

def cosine_loss(a, b):
    return (1 - F.cosine_similarity(a, b)).mean()

def eval_epoch(loader):
    model.eval()
    total_loss, total_cos, n = 0., 0., 0
    with torch.no_grad():
        for z_mae, z_llava in loader:
            z_mae, z_llava = z_mae.to(DEVICE), z_llava.to(DEVICE)
            z_proj = model(z_mae)
            total_loss += cosine_loss(z_proj, z_llava).item() * z_mae.shape[0]
            total_cos  += F.cosine_similarity(z_proj, z_llava).mean().item() * z_mae.shape[0]
            n          += z_mae.shape[0]
    return total_loss / n, total_cos / n

for epoch in range(N_EPOCHS):
    model.train()
    total_loss, n = 0., 0
    for z_mae, z_llava in train_loader:
        z_mae, z_llava = z_mae.to(DEVICE), z_llava.to(DEVICE)
        optimizer.zero_grad()
        loss = cosine_loss(model(z_mae), z_llava)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * z_mae.shape[0]
        n          += z_mae.shape[0]
    scheduler.step()

    train_loss = total_loss / n
    val_loss, val_cos = eval_epoch(val_loader)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_cos'].append(val_cos)

    if val_cos > best_cos:
        best_cos, best_epoch, patience_counter = val_cos, epoch + 1, 0
        torch.save(model.state_dict(), 'wp4_projection_best.pt')
    else:
        patience_counter += 1

    if (epoch + 1) % 20 == 0:
        print(f'Epoch {epoch+1:3d} | train={train_loss:.4f} | '
              f'val={val_loss:.4f} | cos={val_cos:.4f} | patience={patience_counter}/{PATIENCE}')

    if patience_counter >= PATIENCE:
        print(f'Early stopping a l epoch {epoch + 1}')
        break

print(f'Meilleur modele : epoch {best_epoch}, val_cos={best_cos:.4f}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='train')
axes[0].plot(history['val_loss'],   label='val')
axes[0].axvline(best_epoch-1, color='red', linestyle='--', alpha=0.5, label=f'best={best_epoch}')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss cosinus')
axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(history['val_cos'], color='darkorange')
axes[1].axvline(best_epoch-1, color='red', linestyle='--', alpha=0.5, label=f'best={best_cos:.4f}')
axes[1].axhline(1.0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Cos. sim.')
axes[1].set_title('Alignement MAE -> LLaVA'); axes[1].legend()
plt.tight_layout()
plt.savefig('wp4_training_curves.png', dpi=150)
plt.show()


## 5. Setup inférence

Chargement : meilleur f_θ, MAE (float32), LLaVA (vision tower + MLP connector + LLM en fp16), tokenizer, vocabulaire filtré.

In [ ]:
import gc
from transformers import ViTMAEModel, ViTImageProcessor
from transformers import LlavaForConditionalGeneration, CLIPImageProcessor, LlamaTokenizer

# Charger le meilleur modele f_theta
model.load_state_dict(torch.load('wp4_projection_best.pt'))
model.eval()
norm_stats = torch.load('wp4_norm_stats.pt')
z_mean = norm_stats['mean']
z_std  = norm_stats['std']

# MAE en float32
mae_processor = ViTImageProcessor(
    size={'height': 224, 'width': 224},
    image_mean=[0.485, 0.456, 0.406],
    image_std=[0.229, 0.224, 0.225],
)
mae_encoder = ViTMAEModel.from_pretrained('./vit-mae-large').to(DEVICE)
mae_encoder.eval()

# LLaVA complet en fp16
llava_full = LlavaForConditionalGeneration.from_pretrained(
    './llava-1.5-7b-hf', torch_dtype=torch.float16
).to(DEVICE)
llava_full.eval()

llm          = llava_full.language_model
llava_vision = llava_full.vision_tower
llava_mlp    = llava_full.multi_modal_projector
clip_proc    = CLIPImageProcessor.from_pretrained('./llava-1.5-7b-hf')
tokenizer    = LlamaTokenizer.from_pretrained('./llava-1.5-7b-hf', use_fast=False)

print(f'VRAM utilisee : {torch.cuda.memory_allocated() / 1e9:.1f} GB')

# Vocabulaire LLM filtre : vrais mots alphabetiques minuscules longueur > 2
print('Chargement matrice embeddings...')
word_embeddings = llm.get_input_embeddings().weight.detach().float()
word_embeddings = F.normalize(word_embeddings, dim=-1)  # (32000, 4096)

print('Filtrage du vocabulaire...')
VALID_IDS = []
for tid in range(word_embeddings.shape[0]):
    try:
        w = tokenizer.decode([tid]).strip()
        if w and len(w) > 2 and w.replace(' ', '').isalpha() and w.islower():
            VALID_IDS.append(tid)
    except Exception:
        continue
valid_embeddings = word_embeddings[VALID_IDS]
print(f'Vocabulaire filtre : {len(VALID_IDS)} mots sur {word_embeddings.shape[0]}')


## 6. Fonctions utilitaires

In [ ]:
def encode_masked(image_pil, seed=42):
    """Encodage MAE avec masquage aleatoire 75%. Retourne tokens, masque, positions."""
    inputs = mae_processor(images=image_pil, return_tensors='pt')
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    gen   = torch.Generator().manual_seed(seed)
    noise = torch.rand(1, 196, generator=gen).to(DEVICE)
    with torch.no_grad():
        out = mae_encoder(**inputs, noise=noise)
    patch_tokens = out.last_hidden_state[0, 1:]         # (49, 1024)
    mask         = out.mask[0]                           # (196,)
    visible_ids  = torch.where(mask == 0)[0].tolist()   # positions originales
    return patch_tokens, mask, visible_ids


def encode_full(image_pil):
    """Encodage MAE sans masquage. Retourne 196 tokens."""
    inputs = mae_processor(images=image_pil, return_tensors='pt')
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        out = mae_encoder(**inputs, noise=torch.zeros(1, 196).to(DEVICE))
    return out.last_hidden_state[0, 1:]  # (196, 1024)


def project_token(z_i):
    """Normalise et projette un token MAE via f_theta. Retourne (4096,)."""
    z_norm = (z_i.cpu() - z_mean) / z_std
    with torch.no_grad():
        z_proj = model(z_norm.unsqueeze(0).to(DEVICE))[0]
    return z_proj.cpu().float()


def extract_patch(image_pil, patch_id, patch_size=16, grid_size=14):
    """Extrait le crop 16x16 du patch_id dans une image 224x224."""
    row = patch_id // grid_size
    col = patch_id  % grid_size
    return image_pil.crop((col*patch_size, row*patch_size,
                           (col+1)*patch_size, (row+1)*patch_size))


def top_k_words(z_proj, k=3):
    """k mots les plus proches dans le vocabulaire LLM filtre."""
    z       = F.normalize(z_proj.cpu().float(), dim=-1)
    scores  = valid_embeddings @ z
    top_ids = scores.topk(k).indices.tolist()
    return [tokenizer.decode([VALID_IDS[i]]).strip() for i in top_ids]


# Template Vicuna obligatoire pour LLaVA-1.5
SYSTEM    = ('A chat between a curious user and an artificial intelligence assistant. '
             'The assistant gives helpful, detailed, and polite answers to the user questions.')
USER_TEXT = 'Describe this image in one sentence.'
BEFORE    = f'{SYSTEM} USER: '
AFTER     = f'\n{USER_TEXT} ASSISTANT:'

before_ids    = tokenizer(BEFORE, return_tensors='pt', add_special_tokens=True).input_ids.to(DEVICE)
after_ids     = tokenizer(AFTER,  return_tensors='pt', add_special_tokens=False).input_ids.to(DEVICE)
before_embeds = llm.get_input_embeddings()(before_ids).half()  # (1, N, 4096)
after_embeds  = llm.get_input_embeddings()(after_ids).half()   # (1, M, 4096)


def llm_describe(visual_tokens_4096):
    """
    Genere une description depuis une sequence de tokens visuels (4096 dim).
    visual_tokens_4096 : Tensor (N, 4096) en float32 ou float16
    Structure : [SYSTEM USER:] + [visual tokens] + [question ASSISTANT:]
    """
    visual = visual_tokens_4096.half().to(DEVICE).unsqueeze(0)  # (1, N, 4096)
    inputs_embeds = torch.cat([before_embeds, visual, after_embeds], dim=1)
    with torch.no_grad():
        output_ids = llm.generate(
            inputs_embeds=inputs_embeds,
            max_new_tokens=50,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()


print('Fonctions utilitaires definies')


## 7. Sanity check — MAE full encoding vs LLaVA natif

On compare les descriptions générées par :
- **LLaVA natif** : CLIP encoder (576 tokens) → MLP connector → LLM
- **MAE → f_θ** : MAE full encoding (196 tokens) → projeter chacun via f_θ → LLM

Si f_θ est bien aligné, les descriptions devraient être sémantiquement proches.

In [ ]:
from datasets import load_dataset

ds = load_dataset('parquet', data_files={
    'validation': './imagenet100/data/validation-*.parquet',
})


def describe_llava_native(image_pil):
    """Description via le pipeline LLaVA natif (576 tokens CLIP)."""
    inputs = clip_proc(images=image_pil, return_tensors='pt', do_rescale=True)
    pix    = inputs['pixel_values'].to(DEVICE).half()
    with torch.no_grad():
        vis_out = llava_vision(pix)
        patches = vis_out.last_hidden_state[:, 1:]    # (1, 576, 1024)
        visual  = llava_mlp(patches)[0]               # (576, 4096)
    return llm_describe(visual)


def describe_mae_projected(image_pil):
    """Description via MAE full encoding + projection f_theta (196 tokens)."""
    image_224    = image_pil.resize((224, 224))
    patch_tokens = encode_full(image_224)              # (196, 1024)
    projected    = torch.stack([project_token(t) for t in patch_tokens])  # (196, 4096)
    return llm_describe(projected)


N_IMAGES = 5
print(f'{"Classe":<38} {"LLaVA natif":<45} {"MAE -> f_theta"}')
print('-' * 120)

for i in range(N_IMAGES):
    item      = ds['validation'][i]
    image_pil = item['image'].convert('RGB')
    label     = item['text'][:35]

    desc_llava = describe_llava_native(image_pil)
    desc_mae   = describe_mae_projected(image_pil)

    print(f'{label:<38} {desc_llava:<45} {desc_mae}')


## 8. Visualisation patch-level

On encode une image en mode **masqué** et on analyse les tokens visibles.

In [ ]:
IMG_IDX = 0
SEED    = 42

item      = ds['validation'][IMG_IDX]
image_pil = item['image'].convert('RGB').resize((224, 224))
label_txt = item['text']
print(f'Classe : {label_txt}')

patch_tokens, mask, visible_ids = encode_masked(image_pil, seed=SEED)
print(f'Patches visibles : {len(visible_ids)}')

# Projection + top-k mots pour chaque patch visible
results = []
for token, patch_id in zip(patch_tokens, visible_ids):
    z_proj     = project_token(token)
    words      = top_k_words(z_proj, k=3)
    patch_crop = extract_patch(image_pil, patch_id)
    results.append({'patch_id': patch_id, 'crop': patch_crop, 'words': words})

# Figure 1 : image avec masque
fig_img, ax = plt.subplots(1, 1, figsize=(4, 4))
overlay = np.array(image_pil).copy().astype(float)
for pid in range(196):
    if pid not in visible_ids:
        r, c = pid // 14, pid % 14
        overlay[r*16:(r+1)*16, c*16:(c+1)*16] *= 0.15
ax.imshow(overlay.astype(np.uint8))
ax.set_title(f'{label_txt}', fontsize=9)
ax.axis('off')
plt.tight_layout()
plt.savefig('wp4_image_masque.png', dpi=150, bbox_inches='tight')
plt.show()

# Figure 2 : grille 7x7 des patches visibles avec leurs mots
COLS  = 7
ROWS  = (len(results) + COLS - 1) // COLS
fig, axes = plt.subplots(ROWS, COLS, figsize=(COLS * 2.2, ROWS * 2.8))
fig.suptitle(f'Patches visibles MAE + top-3 mots LLM — {label_txt}', fontsize=11)
axes = axes.flatten()

for idx, res in enumerate(results):
    ax = axes[idx]
    ax.imshow(res['crop'].resize((64, 64), resample=0))
    ax.set_title(f'p{res["patch_id"]}', fontsize=7, pad=2)
    ax.axis('off')
    words_str = ' | '.join(res['words']) if res['words'] else 'n/a'
    ax.text(0.5, -0.08, words_str, transform=ax.transAxes,
            fontsize=5.5, ha='center', va='top', color='#333333')

for idx in range(len(results), len(axes)):
    axes[idx].axis('off')

plt.subplots_adjust(hspace=0.45, wspace=0.15)
plt.savefig('wp4_patch_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figures sauvegardees')


## 9. Description LLM par patch visible

On injecte chaque token patch projeté individuellement dans le LLM avec le template Vicuna.

In [ ]:
N_TEST = 10
step   = max(1, len(results) // N_TEST)
subset = results[::step][:N_TEST]

fig, axes = plt.subplots(2, N_TEST, figsize=(N_TEST * 2.5, 5))
fig.suptitle(f'Description LLM par patch — {label_txt}', fontsize=13)

for col, res in enumerate(subset):
    token_idx = visible_ids.index(res['patch_id'])
    z_proj    = project_token(patch_tokens[token_idx])  # (4096,)
    desc      = llm_describe(z_proj.unsqueeze(0))        # 1 seul token visuel

    axes[0, col].imshow(res['crop'].resize((64, 64), resample=0))
    axes[0, col].set_title(f'p{res["patch_id"]}', fontsize=7)
    axes[0, col].axis('off')
    axes[1, col].text(0.5, 0.5, desc[:60], ha='center', va='center',
                      fontsize=6, wrap=True, transform=axes[1, col].transAxes)
    axes[1, col].axis('off')
    print(f'Patch {res["patch_id"]:3d} : {desc}')

plt.tight_layout()
plt.savefig('wp4_llm_descriptions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure sauvegardee -> wp4_llm_descriptions.png')
